# BSI (reduced) in NHANES — Exercises 1–6: baselines, seeds, ablation, Model Card, Datasheet, scorecard
Doctoral Seminar R6S10714 (Dr. Loveleen Gaur) · Class 7 · Aly Gallo López

**What this notebook does.** It re-runs, end to end and from public CDC files, the analysis of `BSI_NHANES_v4_1_tiempo_hasta_evento` (does a reduced Biological State Index add to age + sex + epigenetic clock + allostatic load for all-cause mortality?). On top of it, it runs Exercises 1–3 (tuned baseline, seeds, ablation) plus a subgroup breakdown, and **generates** `MODEL_CARD.md` (Ex. 4) and `DATASHEET.md` (Ex. 5) with every number taken from the results above, then scores them (Ex. 6).

**How to run:** *Runtime → Run all* (about 15–25 minutes on free CPU). Nothing has to be uploaded; all data are downloaded from the CDC.

**Honest scope.** The BSI here is *reduced* (inflammation + metabolism + epigenetic-age acceleration; no cortisol, no BDNF) and the data are a single cross-sectional snapshot with mortality follow-up. This does **not** test Adaptive Capacity (PE3). Exercises 1–3 are **exploratory**: they were designed after the primary analysis was fixed and they reuse the same data. The MIDUS 3 evaluation is documented in the cards from its own notebook (data under an ICPSR data-use agreement, not redistributed, not re-run here).

**Mapping:** Ex. 1 = §2 · Ex. 2 = §3 · Ex. 3 = §4 · subgroups = §5 · Ex. 4 = §6 · Ex. 5 = §7 · Ex. 6 = §8.

In [1]:
!pip -q install lifelines

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 1.8 MB/s eta 0:00:00


In [2]:
%%writefile bsi_nhanes.py
"""Reduced BSI helpers (NHANES 1999-2002, age >= 50). Same definitions as notebooks v3 / v4.1; nothing in the index was changed."""
import re
import numpy as np
import pandas as pd

SEED = 2026
CANDIDATES = {
    "age": ["RIDAGEYR"], "sex": ["RIAGENDR"],
    "crp": ["LBXCRP", "LBDCRP"], "glucose": ["LBXGLU"], "insulin": ["LBXIN"],
    "tg": ["LBXTR"], "hdl": ["LBDHDL", "LBXHDD"], "hba1c": ["LBXGH"],
    "uma": ["URXUMA"], "ucr": ["URXUCR"],
}
CLOCK_REGEX = r"(pheno|grim|dunedin|hannum|horvath)"

def pick(df, names):
    for n in names:
        if n in df.columns:
            return n
    low = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in low:
            return low[n.lower()]
    return None

def check_columns(df):
    found, missing = {}, []
    for role, names in CANDIDATES.items():
        c = pick(df, names)
        (found.__setitem__(role, c) if c else missing.append(role))
    clocks = [c for c in df.columns if re.search(CLOCK_REGEX, c, re.I)]
    return found, missing, clocks

def zs(x):
    return (x - x.mean()) / x.std(ddof=0)

def build_scores(df, cols, clock_col):
    d = pd.DataFrame({"seqn": df["SEQN"]})
    d["age"] = df[cols["age"]]; d["female"] = (df[cols["sex"]] == 2).astype(int)
    crp, ins = np.log(df[cols["crp"]].clip(lower=1e-3)), np.log(df[cols["insulin"]].clip(lower=0.1))
    tg, glu = np.log(df[cols["tg"]].clip(lower=1)), df[cols["glucose"]]
    hba, hdl = df[cols["hba1c"]], df[cols["hdl"]]
    acr = np.log((df[cols["uma"]].clip(lower=0.1) / df[cols["ucr"]].clip(lower=1)) * 100)
    # epigenetic clock acceleration = residual of the clock on chronological age
    c = df[clock_col]; ok = c.notna() & d["age"].notna()
    slope, icpt = np.polyfit(d.loc[ok, "age"], c[ok], 1)
    accel = c - (icpt + slope * d["age"])
    # domains (higher = worse)
    d["dom_inflam"] = zs(pd.concat([zs(crp), zs(acr)], axis=1).mean(axis=1))
    d["dom_metab"] = zs(pd.concat([zs(glu), zs(ins), zs(tg), zs(hba), -zs(hdl)], axis=1).mean(axis=1))
    d["dom_epi"] = zs(accel)
    dom = d[["dom_inflam", "dom_metab", "dom_epi"]]
    d["bsi_mean"] = dom.mean(axis=1)    # primary: equal-weight mean of the 3 domains
    d["bsi_max"] = dom.max(axis=1)      # secondary: most altered domain
    d["clock_accel"] = accel
    # simplified allostatic load: number of the 7 markers in the worst quartile
    risk = pd.concat([crp > crp.quantile(.75), acr > acr.quantile(.75), hba > hba.quantile(.75),
                      tg > tg.quantile(.75), hdl < hdl.quantile(.25),
                      glu > glu.quantile(.75), ins > ins.quantile(.75)], axis=1)
    d["allo_load"] = risk.sum(axis=1)
    return d


Writing bsi_nhanes.py


In [3]:
# 1) Download public files from the CDC (each file is validated)
import os, glob, requests, numpy as np, pandas as pd
import bsi_nhanes as B
os.makedirs('data', exist_ok=True)
for f in glob.glob('data/*'): os.remove(f)

BASE_URL = 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{y}/DataFiles/{n}.xpt'
NAMES = {
 '1999': ['DEMO','LAB11','LAB10AM','LAB13AM','LAB13','LAB10','LAB16'],
 '2001': ['DEMO_B','L11_B','L10AM_B','L13AM_B','L13_B','L10_B','L16_B'],
}
MORT = 'https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/linked_mortality/'
MORTFILES = ['NHANES_1999_2000_MORT_2019_PUBLIC.dat','NHANES_2001_2002_MORT_2019_PUBLIC.dat']
DNAM_URL = 'https://wwwn.cdc.gov/nchs/data/nhanes/dnam/dnmepi.sas7bdat'

def es_xport(b): return b[:20].startswith(b'HEADER RECORD')
fallan = []
def bajar(url, dest, valido):
    r = requests.get(url, timeout=120)
    if r.status_code == 200 and valido(r.content):
        open(dest, 'wb').write(r.content); print('OK   ', os.path.basename(dest), len(r.content), 'bytes')
    else:
        fallan.append(os.path.basename(dest)); print('FAIL ', os.path.basename(dest), r.status_code, r.content[:40])

for y, ns in NAMES.items():
    for n in ns: bajar(BASE_URL.format(y=y, n=n), f'data/{n}.XPT', es_xport)
for n in MORTFILES:
    bajar(MORT + n, f'data/{n}', lambda b: len(b) > 100000 and not b[:15].lower().startswith(b'<!doc'))
bajar(DNAM_URL, 'data/dnmepi.sas7bdat', lambda b: len(b) > 100000 and not b[:15].lower().startswith(b'<!doc'))
print('\nFailed:', fallan)
assert not fallan, 'A download failed; stop here and send me this list.'


OK    DEMO.XPT 11500560 bytes
OK    LAB11.XPT 469040 bytes
OK    LAB10AM.XPT 210960 bytes
OK    LAB13AM.XPT 221040 bytes
OK    LAB13.XPT 335200 bytes
OK    LAB10.XPT 109200 bytes
OK    LAB16.XPT 314240 bytes
OK    DEMO_B.XPT 3273520 bytes
OK    L11_B.XPT 446240 bytes
OK    L10AM_B.XPT 236480 bytes
OK    L13AM_B.XPT 251760 bytes
OK    L13_B.XPT 371920 bytes
OK    L10_B.XPT 120160 bytes
OK    L16_B.XPT 347360 bytes
OK    NHANES_1999_2000_MORT_2019_PUBLIC.dat 487666 bytes
OK    NHANES_2001_2002_MORT_2019_PUBLIC.dat 540362 bytes
OK    dnmepi.sas7bdat 1245184 bytes

Failed: []


In [4]:
# 2) Merge cycles, DNA-methylation clocks and linked mortality; keep age >= 50
def read_cycle(names):
    base = None
    for n in names:
        t = pd.read_sas(f'data/{n}.XPT', format='xport', encoding='utf-8')
        base = t if base is None else base.merge(t, on='SEQN', how='outer', suffixes=('','_dup'))
    return base
lab = pd.concat([read_cycle(NAMES['1999']), read_cycle(NAMES['2001'])], ignore_index=True)
epi = pd.read_sas('data/dnmepi.sas7bdat', encoding='utf-8')
epi.columns = [c if c == 'SEQN' else 'epi_' + c for c in epi.columns]
cols, miss, clocks = B.check_columns(lab.merge(epi, on='SEQN'))
CLOCK = 'epi_PhenoAge'   # fixed before seeing any result
assert not miss and CLOCK in epi.columns, 'A variable or the clock is missing.'

spec = [(0,14),(14,15),(15,16),(16,19),(43,46),(46,49)]
names = ['SEQN','ELIGSTAT','MORTSTAT','UCOD','PERMTH_INT','PERMTH_EXM']
m = pd.concat([pd.read_fwf(f'data/{f}', colspecs=spec, names=names) for f in MORTFILES], ignore_index=True)
for c in ['SEQN','ELIGSTAT','MORTSTAT','PERMTH_EXM','PERMTH_INT']:
    m[c] = pd.to_numeric(m[c], errors='coerce')
m = m[m['ELIGSTAT'] == 1]
assert set(m['MORTSTAT'].dropna().unique()) <= {0,1}
df = lab.merge(epi, on='SEQN').merge(m[['SEQN','MORTSTAT','PERMTH_EXM','PERMTH_INT']], on='SEQN')
df = df[df[cols['age']] >= 50].copy()
df['PERMTH_EXM'] = df['PERMTH_EXM'].fillna(df['PERMTH_INT'])
df.loc[df['PERMTH_EXM'] == 0, 'PERMTH_EXM'] = 0.5     # same-month deaths kept, coded 0.5 month
print('Participants >= 50 with methylation and linked mortality:', len(df), '| deaths:', int(df['MORTSTAT'].sum()))


Participants >= 50 with methylation and linked mortality: 4446 | deaths: 2493


In [5]:
# 3) Cox helpers (out-of-fold risk, concordance index, paired bootstrap of the C-index difference)
import warnings, importlib.metadata as imd
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from sklearn.model_selection import StratifiedKFold
warnings.filterwarnings("ignore")

def cox_oof(d, feats_by_model, folds=5, repeats=5, seed0=None, pen=1e-4):
    """Out-of-fold log partial hazard; same folds for every model."""
    seed0 = B.SEED if seed0 is None else seed0
    y = d["event"].values
    out = {k: np.zeros((repeats, len(d))) for k in feats_by_model}
    for r in range(repeats):
        for tr, te in StratifiedKFold(folds, shuffle=True, random_state=seed0 + r).split(d, y):
            for k, f in feats_by_model.items():
                cph = CoxPHFitter(penalizer=pen)
                cph.fit(d.iloc[tr][f + ["time", "event"]], duration_col="time", event_col="event")
                out[k][r, te] = cph.predict_log_partial_hazard(d.iloc[te][f]).values
    return {k: v.mean(axis=0) for k, v in out.items()}

def cidx(t, e, risk): return concordance_index(t, -risk, e)

def boot_dC(t, e, r_new, r_old, n_boot=2000, seed=B.SEED):
    rng = np.random.default_rng(seed); n = len(t); ds = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        if e[i].sum() < 5: continue
        ds.append(cidx(t[i], e[i], r_new[i]) - cidx(t[i], e[i], r_old[i]))
    return np.percentile(ds, [2.5, 97.5])

def verdict(lo, hi, margin=0.01):
    return "ADDS" if lo > 0 else "DOES NOT ADD" if (lo > -margin and hi < margin) else "INCONCLUSIVE"

VERSIONS = {p: imd.version(p) for p in ["lifelines", "scikit-learn", "pandas", "numpy"]}
import sys; VERSIONS["python"] = sys.version.split()[0]
print(VERSIONS)


{'lifelines': '0.30.3', 'scikit-learn': '1.6.1', 'pandas': '2.2.3', 'numpy': '2.1.3', 'python': '3.13.16'}


## 1. Primary analysis (re-run of v4.1) — the numbers the cards will quote
Models (Cox, 5-fold cross-validation repeated 5 times, same folds for all models): M1 age+sex · M2 +epigenetic clock · M3 +allostatic load · M4 +BSI (equal-weight mean of 3 domains). Primary contrast: ΔC = C(M4) − C(M3), paired bootstrap 95% CI. Decision rule fixed beforehand: ADDS if the lower limit > 0; DOES NOT ADD if the CI lies inside ±0.01; otherwise INCONCLUSIVE.

In [6]:
d = B.build_scores(df, cols, CLOCK)
d["event"] = df["MORTSTAT"].values; d["time"] = df["PERMTH_EXM"].values
d["w"] = df["epi_WTDN4YR"].values if "epi_WTDN4YR" in df.columns else np.nan
d["race"] = df["RIDRETH1"].values if "RIDRETH1" in df.columns else np.nan
d_all = d.copy()
d = d.dropna(subset=[c for c in d.columns if c not in ("w", "race")]).reset_index(drop=True)
assert (d["time"] > 0).all()
t, e = d["time"].values, d["event"].astype(int).values

BASE = ["age", "female", "clock_accel"]
FE = {"M1 age+sex": ["age", "female"], "M2 +clock": BASE, "M3 +allostatic load": BASE + ["allo_load"],
      "M4 +BSI (mean)": BASE + ["allo_load", "bsi_mean"]}
R = cox_oof(d, FE)
tabla = pd.DataFrame([{"model": k, "C (out-of-fold)": round(cidx(t, e, v), 4)} for k, v in R.items()])
display(tabla)
lo, hi = boot_dC(t, e, R["M4 +BSI (mean)"], R["M3 +allostatic load"])
dC = cidx(t, e, R["M4 +BSI (mean)"]) - cidx(t, e, R["M3 +allostatic load"])
RES = {"n": len(d), "deaths": int(e.sum()), "C": {k: float(cidx(t, e, v)) for k, v in R.items()},
       "dC": float(dC), "dC_ci": (float(lo), float(hi)), "dC_verdict": verdict(lo, hi)}
print(f"PRIMARY  M4 - M3: dC = {dC:.4f}  95% CI ({lo:.4f}, {hi:.4f}) -> {RES['dC_verdict']}   [n={RES['n']}, deaths={RES['deaths']}]")

# hazard ratio of the BSI per 1 SD, adjusted; proportional-hazards check
dd = d.copy(); dd["bsi_z"] = (dd["bsi_mean"] - dd["bsi_mean"].mean()) / dd["bsi_mean"].std()
f4 = BASE + ["allo_load", "bsi_z"]
cph = CoxPHFitter(penalizer=1e-4).fit(dd[f4 + ["time", "event"]], "time", "event")
s = cph.summary.loc["bsi_z"]
RES["HR"] = (float(s["exp(coef)"]), float(s["exp(coef) lower 95%"]), float(s["exp(coef) upper 95%"]))
try:
    from lifelines.statistics import proportional_hazard_test
    RES["PH_p_bsi"] = float(proportional_hazard_test(cph, dd[f4 + ["time", "event"]], time_transform="rank").summary.loc["bsi_z", "p"])
except Exception as ex:
    RES["PH_p_bsi"] = None; print("PH test failed:", ex)
print("HR per 1 SD of BSI: %.3f (%.3f, %.3f); PH test p = %s" % (*RES["HR"], RES["PH_p_bsi"]))

# survey-weighted sensitivity (approximate: WTDN4YR belongs to the methylation subsample)
RES["HR_w"] = None
if dd["w"].notna().any():
    dw = dd[dd["w"] > 0]
    sw = CoxPHFitter(penalizer=1e-4).fit(dw[f4 + ["time", "event", "w"]], "time", "event", weights_col="w", robust=True).summary.loc["bsi_z"]
    RES["HR_w"] = (float(sw["exp(coef)"]), float(sw["exp(coef) lower 95%"]), float(sw["exp(coef) upper 95%"]))
    print("Weighted HR: %.3f (%.3f, %.3f)" % RES["HR_w"])


,model,C (out-of-fold)
0,M1 age+sex,0.7078
1,M2 +clock,0.7124
2,M3 +allostatic load,0.7161
3,M4 +BSI (mean),0.7232


PRIMARY  M4 - M3: dC = 0.0071  95% CI (0.0033, 0.0108) -> ADDS   [n=2530, deaths=1359]
HR per 1 SD of BSI: 1.332 (1.207, 1.471); PH test p = 0.0017531758662935922
Weighted HR: 1.299 (1.091, 1.547)


In [7]:
# Complete cases + control: BSI versus the SAME 7 markers entered as continuous variables
dc = d_all.copy()
dc["l_crp"] = np.log(df[cols["crp"]].clip(lower=1e-3)).values
dc["l_acr"] = np.log((df[cols["uma"]].clip(lower=0.1) / df[cols["ucr"]].clip(lower=1)) * 100).values
dc["hba1c"] = df[cols["hba1c"]].values; dc["l_tg"] = np.log(df[cols["tg"]].clip(lower=1)).values
dc["hdl"] = df[cols["hdl"]].values; dc["glu"] = df[cols["glucose"]].values
dc["l_ins"] = np.log(df[cols["insulin"]].clip(lower=0.1)).values
dc = dc.drop(columns=["w", "race"]).dropna().reset_index(drop=True)
tc, ec = dc["time"].values, dc["event"].astype(int).values
MK = ["l_crp", "l_acr", "hba1c", "l_tg", "hdl", "glu", "l_ins"]
FE2 = {"M3": BASE + ["allo_load"], "M4": BASE + ["allo_load", "bsi_mean"], "M6 continuous markers": BASE + ["allo_load"] + MK}
R2 = cox_oof(dc, FE2)
lo1, hi1 = boot_dC(tc, ec, R2["M4"], R2["M3"]); d1 = cidx(tc, ec, R2["M4"]) - cidx(tc, ec, R2["M3"])
lo2, hi2 = boot_dC(tc, ec, R2["M4"], R2["M6 continuous markers"]); d2 = cidx(tc, ec, R2["M4"]) - cidx(tc, ec, R2["M6 continuous markers"])
RES["cc"] = {"n": len(dc), "deaths": int(ec.sum()), "dC": float(d1), "ci": (float(lo1), float(hi1))}
RES["ctrl"] = {"dC": float(d2), "ci": (float(lo2), float(hi2)), "verdict": verdict(lo2, hi2)}
print(f"Complete cases (n={len(dc)}, deaths={int(ec.sum())}): M4-M3 dC = {d1:.4f} ({lo1:.4f}, {hi1:.4f})")
print(f"CONTROL  BSI over continuous markers (M4-M6): dC = {d2:.4f} ({lo2:.4f}, {hi2:.4f}) -> {RES['ctrl']['verdict']}")


Complete cases (n=1228, deaths=644): M4-M3 dC = 0.0054 (0.0015, 0.0095)
CONTROL  BSI over continuous markers (M4-M6): dC = -0.0025 (-0.0088, 0.0039) -> DOES NOT ADD


## 2. Exercise 1 — Weak versus tuned baseline
**Question:** does the BSI gain survive when the baseline is tuned as hard as the "new" model? **Fair protocol:** the same folds, the same hyper-parameter grid (Cox ridge penalizer) and the same budget (5 values × 3 repeats × 5 folds) for the baseline (M3) and for the model with the BSI (M4). The weak baseline is age + sex with the default penalizer. **Limit:** the penalizer is picked on the same out-of-fold predictions, so both models are slightly optimistic; this affects both equally.

In [8]:
GRID = [0.0, 1e-4, 1e-3, 1e-2, 1e-1]
rows = []
for pen in GRID:
    Rp = cox_oof(d, {"M1": FE["M1 age+sex"], "M3": FE["M3 +allostatic load"], "M4": FE["M4 +BSI (mean)"]}, repeats=3, pen=pen)
    rows.append({"penalizer": pen, "C_M1": cidx(t, e, Rp["M1"]), "C_M3": cidx(t, e, Rp["M3"]), "C_M4": cidx(t, e, Rp["M4"])})
G = pd.DataFrame(rows); display(G.round(4))
best3 = G.loc[G.C_M3.idxmax()]; best4 = G.loc[G.C_M4.idxmax()]
default = G[G.penalizer == 1e-4].iloc[0]
T1 = pd.DataFrame([
    {"comparison": "BSI model (M4, default) vs WEAK baseline (M1 age+sex, default)", "gain in C": default.C_M4 - default.C_M1},
    {"comparison": "BSI model (M4, default) vs baseline M3 (default)", "gain in C": default.C_M4 - default.C_M3},
    {"comparison": f"M4 tuned (pen={best4.penalizer}) vs M3 tuned (pen={best3.penalizer})", "gain in C": best4.C_M4 - best3.C_M3}])
display(T1.round(4))
RES["ex1"] = {"weak": float(default.C_M4 - default.C_M1), "default": float(default.C_M4 - default.C_M3),
              "tuned": float(best4.C_M4 - best3.C_M3), "span": float(max(G.C_M3.max() - G.C_M3.min(), G.C_M4.max() - G.C_M4.min())), "pen3": float(best3.penalizer), "pen4": float(best4.penalizer)}
print("Reading: the gain over the weak baseline is mostly the allostatic load, the clock and age; the BSI-specific gain is the last row.")


,penalizer,C_M1,C_M3,C_M4
0,0.0000,0.7078,0.7163,0.7233
1,0.0001,0.7078,0.7163,0.7233
2,0.0010,0.7078,0.7163,0.7233
3,0.0100,0.7078,0.7163,0.7233
4,0.1000,0.7077,0.7164,0.7230


,comparison,gain in C
0,"BSI model (M4, default) vs WEAK baseline (M1 a...",0.0155
1,"BSI model (M4, default) vs baseline M3 (default)",0.0070
2,M4 tuned (pen=0.001) vs M3 tuned (pen=0.1),0.0069


Reading: the gain over the weak baseline is mostly the allostatic load, the clock and age; the BSI-specific gain is the last row.


## 3. Exercise 2 — Seeds and variance
Same models, 10 seeds (each seed changes the cross-validation folds). We report the BSI gain ΔC = C(M4) − C(M3) for each seed, the mean ± SD, and never only the best seed.

In [9]:
SEEDS = list(range(2026, 2036))
rows = []
for sd in SEEDS:
    Rs = cox_oof(d, {"M3": FE["M3 +allostatic load"], "M4": FE["M4 +BSI (mean)"]}, repeats=1, seed0=sd)
    c3, c4 = cidx(t, e, Rs["M3"]), cidx(t, e, Rs["M4"]); rows.append({"seed": sd, "C_M3": c3, "C_M4": c4, "dC": c4 - c3})
S = pd.DataFrame(rows); display(S.round(4))
RES["ex2"] = {"mean": float(S.dC.mean()), "sd": float(S.dC.std(ddof=1)), "min": float(S.dC.min()), "max": float(S.dC.max()),
              "pos": int((S.dC > 0).sum()), "n_seeds": len(S)}
print("dC over %d seeds: mean %.4f +/- SD %.4f (min %.4f, max %.4f); positive in %d/%d seeds" % (len(S), RES["ex2"]["mean"], RES["ex2"]["sd"], RES["ex2"]["min"], RES["ex2"]["max"], RES["ex2"]["pos"], len(S)))
print("Reporting only the best seed would give dC = %.4f; the mean is %.4f." % (S.dC.max(), S.dC.mean()))


,seed,C_M3,C_M4,dC
0,2026,0.7160,0.7232,0.0072
1,2027,0.7165,0.7232,0.0067
2,2028,0.7162,0.7229,0.0066
3,2029,0.7159,0.7231,0.0072
4,2030,0.7153,0.7225,0.0073
5,2031,0.7154,0.7221,0.0067
6,2032,0.7155,0.7226,0.0071
7,2033,0.7155,0.7227,0.0072
8,2034,0.7160,0.7231,0.0071
9,2035,0.7148,0.7225,0.0076


dC over 10 seeds: mean 0.0071 +/- SD 0.0003 (min 0.0066, max 0.0076); positive in 10/10 seeds
Reporting only the best seed would give dC = 0.0076; the mean is 0.0071.


## 4. Exercise 3 — Ablation of the BSI domains
One component changes per variant and everything else stays fixed: the BSI is recomputed as the mean of the remaining two domains (inflammation, metabolism, epigenetic-age acceleration). Each variant is compared with M3 over the same 10 seeds; we report mean ± SD of ΔC and the drop relative to the full BSI. **Exploratory:** not prespecified. **Important design note:** the epigenetic-clock acceleration is already a covariate in every model, so the BSI's epigenetic domain is a linear function of a covariate. Dropping a domain is therefore algebraically equivalent (up to the tiny ridge penalty) to: *without epigenetic acceleration* = the full BSI (identical by construction), *without metabolism* = inflammation alone, *without inflammation* = metabolism alone. The two single-domain rows are included to show this directly.

In [10]:
DOMS = ["dom_inflam", "dom_metab", "dom_epi"]
variants = {"full BSI (3 domains)": DOMS, "without inflammation": ["dom_metab", "dom_epi"],
            "without metabolism": ["dom_inflam", "dom_epi"], "without epigenetic acceleration": ["dom_inflam", "dom_metab"],
            "inflammation only": ["dom_inflam"], "metabolism only": ["dom_metab"]}
dab = d.copy(); FEA = {"M3": FE["M3 +allostatic load"]}
for i, (name, ds) in enumerate(variants.items()):
    dab["v%d" % i] = dab[ds].mean(axis=1); FEA[name] = BASE + ["allo_load", "v%d" % i]
per_seed = {name: [] for name in variants}
for sd in SEEDS:
    Ra = cox_oof(dab, FEA, repeats=1, seed0=sd); c3 = cidx(t, e, Ra["M3"])
    for name in variants: per_seed[name].append(cidx(t, e, Ra[name]) - c3)
full = np.array(per_seed["full BSI (3 domains)"]); rows = []
for name, v in per_seed.items():
    v = np.array(v); drop = full - v
    rows.append({"variant": name, "dC vs M3 (mean)": v.mean(), "SD": v.std(ddof=1), "drop vs full BSI (mean)": drop.mean(), "SD of drop": drop.std(ddof=1)})
A = pd.DataFrame(rows); display(A.round(4))
RES["ex3"] = {r["variant"]: (float(r["dC vs M3 (mean)"]), float(r["SD"])) for r in rows}
RES["ex3_drop"] = {r["variant"]: (float(r["drop vs full BSI (mean)"]), float(r["SD of drop"])) for r in rows}
print("A component 'helps' only if removing it lowers dC by clearly more than the seed-to-seed SD; the table also reports the components that do not help.")


,variant,dC vs M3 (mean),SD,drop vs full BSI (mean),SD of drop
0,full BSI (3 domains),0.0071,0.0003,0.0000,0.0000
1,without inflammation,-0.0005,0.0004,0.0076,0.0005
2,without metabolism,0.0112,0.0002,-0.0041,0.0003
3,without epigenetic acceleration,0.0071,0.0003,-0.0000,0.0000
4,inflammation only,0.0112,0.0002,-0.0041,0.0003
5,metabolism only,-0.0005,0.0004,0.0076,0.0005


A component 'helps' only if removing it lowers dC by clearly more than the seed-to-seed SD; the table also reports the components that do not help.


## 5. Subgroup breakdown (rubric pillar: disaggregation)
Out-of-fold predictions of the primary analysis (Section 1) evaluated within subgroups. Paired bootstrap, 1,000 replications. Small subgroups give wide intervals: read them as descriptive, not as tests.

In [11]:
groups = {"Sex: male": d.female == 0, "Sex: female": d.female == 1, "Age 50-64": d.age < 65, "Age 65+": d.age >= 65}
rmap = {1: "Mexican American", 2: "Other Hispanic", 3: "Non-Hispanic White", 4: "Non-Hispanic Black", 5: "Other/multiracial"}
if d["race"].notna().any():
    for k, lab_ in rmap.items(): groups["Race/ethnicity: " + lab_] = (d.race == k)
rows = []
for name, mask in groups.items():
    mk = mask.values
    if e[mk].sum() < 30: rows.append({"subgroup": name, "n": int(mk.sum()), "deaths": int(e[mk].sum()), "note": "too few deaths, not estimated"}); continue
    lo_, hi_ = boot_dC(t[mk], e[mk], R["M4 +BSI (mean)"][mk], R["M3 +allostatic load"][mk], n_boot=1000)
    c3, c4 = cidx(t[mk], e[mk], R["M3 +allostatic load"][mk]), cidx(t[mk], e[mk], R["M4 +BSI (mean)"][mk])
    rows.append({"subgroup": name, "n": int(mk.sum()), "deaths": int(e[mk].sum()), "C_M3": c3, "C_M4": c4, "dC": c4 - c3, "CI_low": lo_, "CI_high": hi_})
SG = pd.DataFrame(rows); display(SG.round(4)); RES["subgroups"] = SG.to_dict("records")


,subgroup,n,deaths,C_M3,C_M4,dC,CI_low,CI_high
0,Sex: male,1284,739,0.7120,0.7153,0.0033,-0.0018,0.0083
1,Sex: female,1246,620,0.7162,0.7281,0.0119,0.0065,0.0177
2,Age 50-64,1194,350,0.6099,0.6261,0.0162,0.0011,0.0323
3,Age 65+,1336,1009,0.6503,0.6611,0.0108,0.0046,0.0181
4,Race/ethnicity: Mexican American,720,346,0.6775,0.6926,0.0151,0.0057,0.0251
5,Race/ethnicity: Other Hispanic,163,70,0.7459,0.7531,0.0072,-0.0096,0.0242
6,Race/ethnicity: Non-Hispanic White,1026,603,0.7388,0.7429,0.0040,-0.0003,0.0084
7,Race/ethnicity: Non-Hispanic Black,538,310,0.6932,0.6960,0.0028,-0.0074,0.0143
8,Race/ethnicity: Other/multiracial,83,30,0.7565,0.7585,0.0021,-0.0259,0.0274


## 6. Exercise 4 — Model Card (Mitchell et al., 2019), 9 sections
Every number below is read from `RES` (computed above) except the MIDUS 3 block, which comes from notebook `BSI_MIDUS3_respuesta_al_reto` (not re-run here). Anything we cannot answer is written as *UNKNOWN — to investigate*.

In [12]:
def f4(x): return f"{x:.4f}"
def ci(x): return f"({x[0]:.4f} to {x[1]:.4f})"
pf = lambda p: "n/a" if p is None else f"{p:.4f}"
sgv = [r for r in RES["subgroups"] if pd.notna(r.get("dC"))]
sg_note = (f"Point estimates are positive in {sum(r['dC'] > 0 for r in sgv)} of {len(sgv)} subgroups; the 95% CI excludes zero in: "
           + (", ".join(r["subgroup"] for r in sgv if r["CI_low"] > 0) or "none") + ". These are descriptive: no interaction tests and no multiplicity correction were done, the subgroups are small, and differences between subgroups are not established.")
X3 = RES["ex3"]; inf_only, met_only = X3["inflammation only"][0], X3["metabolism only"][0]
ex3_note = ("Because the epigenetic-clock acceleration is already a covariate in every model, dropping a domain from the BSI is algebraically equivalent (up to the tiny ridge penalty) to: without epigenetic acceleration = the full BSI (identical by construction, so that row carries no information); without metabolism = the inflammation domain alone; without inflammation = the metabolic domain alone. ")
if inf_only > 0.005 and met_only < 0.002:
    ex3_note += (f"Reading: the added value comes from the inflammatory domain (inflammation alone dC = {inf_only:.4f}); the metabolic domain alone adds nothing (dC = {met_only:.4f}), probably because its five markers are already counted in the allostatic load, and it dilutes the equal-weight composite. This is a post hoc exploratory finding: the BSI definition (equal-weight mean of all domains) is NOT changed after seeing it, to avoid cherry-picking and HARKing.")
else:
    ex3_note += "Reading: see the numbers; no domain pattern is claimed beyond them."
sg_lines = "\n".join(
    f"- {r['subgroup']}: n={r['n']}, deaths={r['deaths']}" + (f", dC={r['dC']:.4f} (95% CI {r['CI_low']:.4f} to {r['CI_high']:.4f})" if "dC" in r and r["dC"] == r["dC"] else f" ({r.get('note','')})")
    for r in RES["subgroups"])
ex3 = "\n".join(f"- {k}: dC vs M3 = {v[0]:.4f} +/- {v[1]:.4f} (SD over {RES['ex2']['n_seeds']} seeds); drop vs full BSI = {RES['ex3_drop'][k][0]:.4f} +/- {RES['ex3_drop'][k][1]:.4f}" for k, v in RES["ex3"].items())
E2 = RES["ex2"]; E1 = RES["ex1"]

CARD = {
 "1. Model details": f"""Reduced Biological State Index (BSI) and the Cox proportional-hazards models that test its added value. Author: Aly Gallo Lopez (doctoral student, Deep Tech, UNMSM). Version: notebook v4.1 re-run in this notebook (2026-10). BSI = equal-weight mean of three standardized domain scores: inflammation (log C-reactive protein, log urinary albumin/creatinine ratio), metabolism (glucose, log insulin, log triglycerides, HbA1c, minus HDL) and epigenetic-age acceleration (PhenoAge residual on age). Compared models: M1 age+sex; M2 +epigenetic clock acceleration; M3 +simplified allostatic load; M4 +BSI. Cox models with ridge penalizer 1e-4. Software: {VERSIONS}. License of this code: UNKNOWN — to investigate. Contact: see repository.""",
 "2. Intended use": """Research use only: to test whether a composite of routine biomarkers adds prognostic information for all-cause mortality over age, sex, an epigenetic clock and allostatic load, as an exploratory step of the ChroniCare-AI doctoral project. Out-of-scope / misuse: individual clinical decisions, insurance or employment risk scoring, any claim about Adaptive Capacity (this analysis has no perturbation and no recovery), any claim about biological aging beyond the PhenoAge acceleration used as one input, and use in people younger than 50 or outside the US population sampled.""",
 "3. Relevant factors": """Age and sex are model covariates; race/ethnicity and age group are used only for subgroup reporting. Missingness differs by marker (fasting subsample for insulin, glucose, triglycerides), which can correlate with health status. Instrumentation: NHANES 1999-2002 laboratory assays; DNA methylation measured on Illumina MethylationEPIC v1.0 arrays at Duke University from whole blood stored at -80 C (CDC documentation). The methylation sample is a subsample: about half of the eligible non-Hispanic White participants and all eligible participants of the other groups, so it over-represents non-White groups relative to the US population. Assay comparability with current laboratories is UNKNOWN — to investigate.""",
 "4. Metrics": f"""Concordance index (C) of out-of-fold Cox risk, 5-fold cross-validation repeated 5 times with the same folds for all models. Primary metric: dC = C(M4) - C(M3) with a paired bootstrap 95% CI (2,000 replications). Decision rule fixed before seeing results: ADDS if the lower limit > 0; DOES NOT ADD if the CI lies inside +/-0.01; INCONCLUSIVE otherwise. Also reported: hazard ratio of the BSI per 1 SD, proportional-hazards test, survey-weighted hazard ratio, complete-case analysis and a control against the same markers entered as continuous variables. Calibration and net benefit were not assessed: UNKNOWN — to investigate.""",
 "5. Evaluation data": f"""Same dataset as training, evaluated out-of-fold (no external validation). NHANES 1999-2000 and 2001-2002, adults aged 50 or older in the CDC DNA-methylation release, with linked mortality; primary analysis n={RES['n']} with {RES['deaths']} deaths. See DATASHEET.md. External validation in an independent population: not done (planned for the Rebagliati cohort within the thesis protocol).""",
 "6. Training data": """Same NHANES sample; the Cox models are fitted inside each cross-validation fold. The BSI itself has no fitted weights (equal-weight mean of z-scores, no outcome label used); its standardization parameters and the allostatic-load quartile cut-points are computed in the whole analysed sample, which is a minor source of optimism. No separate held-out set.""",
 "7. Quantitative analyses": f"""Primary (n={RES['n']}, deaths={RES['deaths']}): C(M1)={f4(RES['C']['M1 age+sex'])}, C(M2)={f4(RES['C']['M2 +clock'])}, C(M3)={f4(RES['C']['M3 +allostatic load'])}, C(M4)={f4(RES['C']['M4 +BSI (mean)'])}; dC = {f4(RES['dC'])}, 95% CI {ci(RES['dC_ci'])} -> {RES['dC_verdict']}. Hazard ratio of the BSI per 1 SD, adjusted: {RES['HR'][0]:.3f} ({RES['HR'][1]:.3f} to {RES['HR'][2]:.3f}); proportional-hazards test for the BSI p={pf(RES['PH_p_bsi'])}; survey-weighted HR {RES['HR_w'] if RES['HR_w'] is None else '%.3f (%.3f to %.3f)' % RES['HR_w']} (approximate weights).
Complete cases (n={RES['cc']['n']}, deaths={RES['cc']['deaths']}): dC(M4-M3) = {f4(RES['cc']['dC'])}, 95% CI {ci(RES['cc']['ci'])}. Control against the same 7 markers as continuous variables: dC(M4-M6) = {f4(RES['ctrl']['dC'])}, 95% CI {ci(RES['ctrl']['ci'])} -> {RES['ctrl']['verdict']}: the composite did not outperform its own markers.
Exercise 1 (baselines, same grid for both models): gain of M4 over the weak age+sex baseline = {f4(E1['weak'])}; over M3 with default penalizer = {f4(E1['default'])}; over M3 with both tuned = {f4(E1['tuned'])}. Tuning the ridge penalizer changed C by at most {f4(E1['span'])} for either model, so in this low-dimensional Cox model the result does not depend on tuning; the weak-versus-tuned lesson matters mainly for flexible models (PE1-PE2).
Exercise 2 (seeds): dC over {E2['n_seeds']} seeds = {f4(E2['mean'])} +/- {f4(E2['sd'])} (min {f4(E2['min'])}, max {f4(E2['max'])}); positive in {E2['pos']}/{E2['n_seeds']} seeds. The SD reflects only the assignment of cross-validation folds, not the sampling variability of the data, which is what the bootstrap interval captures.
Exercise 3 (ablation, exploratory):
{ex3}
{ex3_note}
Subgroups (out-of-fold predictions of the primary analysis):
{sg_lines}
{sg_note}
Second evaluation, MIDUS 3 Biomarker Project (ICPSR 38837), from notebook BSI_MIDUS3_respuesta_al_reto, not re-run here: resting BSI (inflammatory, metabolic and neuroendocrine domains, no epigenetic clock) did not improve out-of-fold prediction of heart-rate-variability reactivity to a mental-arithmetic task beyond age, sex, BMI, baseline level, antihypertensive and beta-blocker use and allostatic load (n=521; dR2 +0.0004, 95% CI -0.0082 to 0.0090, DOES NOT ADD). Cortisol reactivity (n=639; dR2 -0.0040) and HRV on standing (n=483; dR2 -0.0017) also showed no gain; systolic-pressure reactivity was not interpretable (out-of-fold R2 below zero).""",
 "8. Ethical considerations": """Public-use, de-identified data; no contact with participants. A mortality-related score can be misused for discrimination (insurance, employment) or be read as a personal prognosis; it must not be used that way. Subgroup performance is reported to avoid hiding differences by sex, age and race/ethnicity, but cells are small. The author is Scientific Director of a company (ADAPTIVE Health OS) for which the broader project is conceived as an engine; this conflict of interest is declared and the null results are reported as findings.""",
 "9. Caveats and recommendations": f"""(1) The gain is small (dC about {f4(RES['dC'])}) and does not exceed the same markers used as continuous variables. (2) The proportional-hazards assumption does not hold for the BSI (p={pf(RES['PH_p_bsi'])}), so a single hazard ratio is an average over follow-up. (3) PhenoAge shares markers with the BSI. (4) The bootstrap interval ignores the variability of model fitting; z-score parameters and quartile cut-points use the whole sample. (5) Reduced BSI: no cortisol, no BDNF. (6) All-cause mortality is the only outcome. (7) One cross-sectional snapshot: this does NOT test Adaptive Capacity. (8) Exercises 1-3 and subgroups are exploratory, run after the primary analysis was fixed, on the same data; the domain ablation is post hoc and does not change the BSI definition. (9) No external validation and no calibration assessment. (10) The CDC documentation instructs analysts to use the sample weights (WTDN4YR) for all analyses; the primary analysis is unweighted and the weighted hazard ratio is only a sensitivity analysis, so the results describe this subsample, not the US population. (11) The public-use linked mortality file perturbs some follow-up times and causes of death with synthetic values (vital status is unchanged), so time-to-event estimates are approximate. Recommendation: do not use the BSI for individual prediction; treat BSI construction as an empirical question (protocol PE1-PE2) and test Adaptive Capacity directly in a perturbation-recovery design (PE3)."""}

def render_card(card):
    out = ["# Model Card — Reduced Biological State Index (BSI), NHANES 1999-2002\n"]
    for k, v in card.items():
        out.append(f"## {k}\n\n{v if v and str(v).strip() else 'UNKNOWN — to investigate'}\n")
    return "\n".join(out)
MODEL_CARD_MD = render_card(CARD)
open("MODEL_CARD.md", "w").write(MODEL_CARD_MD)
print(MODEL_CARD_MD[:3000], "\n...\n[written to MODEL_CARD.md]")


# Model Card — Reduced Biological State Index (BSI), NHANES 1999-2002

## 1. Model details

Reduced Biological State Index (BSI) and the Cox proportional-hazards models that test its added value. Author: Aly Gallo Lopez (doctoral student, Deep Tech, UNMSM). Version: notebook v4.1 re-run in this notebook (2026-10). BSI = equal-weight mean of three standardized domain scores: inflammation (log C-reactive protein, log urinary albumin/creatinine ratio), metabolism (glucose, log insulin, log triglycerides, HbA1c, minus HDL) and epigenetic-age acceleration (PhenoAge residual on age). Compared models: M1 age+sex; M2 +epigenetic clock acceleration; M3 +simplified allostatic load; M4 +BSI. Cox models with ridge penalizer 1e-4. Software: {'lifelines': '0.30.3', 'scikit-learn': '1.6.1', 'pandas': '2.2.3', 'numpy': '2.1.3', 'python': '3.13.16'}. License of this code: UNKNOWN — to investigate. Contact: see repository.

## 2. Intended use

Research use only: to test whether a composite of routine bi

## 7. Exercise 5 — Datasheet (Gebru et al., 2021), 7 sections

In [13]:
SHEET = {
 "1. Motivation": f"""Purpose: to test whether a composite of routine biomarkers (reduced BSI) adds mortality prognosis over age, sex, an epigenetic clock and allostatic load, as an exploratory step of a doctoral project (ChroniCare-AI, UNMSM). We did not create NHANES; we assembled an analysis file from three public CDC releases. Original creators: CDC National Center for Health Statistics (NHANES, DNA-methylation epigenetic biomarkers release, Linked Mortality Files). Funding of the original data: US federal. Funding of this analysis: UNKNOWN — to investigate (self-funded doctoral work).""",
 "2. Composition": f"""Instances: NHANES participants. Cycles 1999-2000 and 2001-2002 (examination files DEMO/DEMO_B and laboratory files), adults aged 50 or older in the CDC DNA-methylation release with eligible linked mortality: 4,446 rows in the merged files (2,493 deaths); the CDC documentation states that participants without adequate blood or DNA, or without consent for future research use, appear in the file with a survey weight of 0; the primary analysis uses n={RES['n']} ({RES['deaths']} deaths), those with a computable clock and BSI inputs; complete-case analysis n={RES['cc']['n']}. Variables used: age, sex, race/ethnicity (subgroups only), CRP, urinary albumin and creatinine, glucose, insulin, triglycerides, HbA1c, HDL, PhenoAge, vital status and follow-up months (to 31 Dec 2019). Missing data: the fasting-dependent markers are missing for many participants; in the primary analysis a missing marker is skipped inside its domain and counted as not-at-risk in the allostatic load; the complete-case analysis requires all seven. Sampling: complex survey; the methylation subsample includes about half of eligible non-Hispanic White participants and all eligible participants of other groups, so it is not nationally representative without the special weights (WTDN4YR). The CDC instructs analysts to use these weights for all analyses; here the primary analysis is unweighted and the weighted analysis is a sensitivity analysis only. Direct identifiers: none in the public-use files. Sensitive attributes: race/ethnicity, health status. Errors and noise: laboratory measurement error; mortality is linked to the National Death Index with a probabilistic (Fellegi-Sunter) algorithm; people without a match are assumed alive; the public-use file perturbs some follow-up times or causes of death with synthetic values (vital status unchanged); imputed probe values and removed samples in the methylation preprocessing make results depend on the CDC preprocessing choices.""",
 "3. Collection process": """Collected by the CDC/NCHS in the NHANES examinations of 1999-2002 (blood and urine); DNA was extracted from whole blood stored at -80 C and methylation was measured with Illumina MethylationEPIC v1.0 arrays in the laboratory of Dr. Yongmei Liu at Duke University, then released by the CDC as a derived epigenetic-biomarker file (documentation first published 31 July 2024); mortality was ascertained by NCHS linkage to the National Death Index with follow-up to 31 December 2019. Only participants who consented to future research use of samples have positive weights in the methylation file (CDC documentation). Collection time frame: 1999-2002 with follow-up to 2019. We did not collect data ourselves. Details of laboratory protocols: see the CDC documentation; not re-verified here.""",
 "4. Preprocessing / cleaning / labeling": """Cycles merged by participant ID; age >= 50; mortality files read with fixed-width positions (validated: only values 0/1); follow-up in months from the examination (from the interview if missing; deaths in month 0 coded 0.5 month, not dropped); markers log-transformed (CRP, insulin, triglycerides, albumin/creatinine ratio) with lower clipping; domain scores are z-scores averaged within domain; epigenetic acceleration = residual of PhenoAge on age; allostatic load = number of 7 markers in the worst quartile. Raw files are downloaded again at each run and are not stored in the repository. Code: this notebook and bsi_nhanes.py.""",
 "5. Uses": """Used for: exploratory added-value analysis of the reduced BSI (this repository). Appropriate: reproducing this analysis; methodological teaching. Should NOT be used for: individual risk prediction or clinical decisions, claims about Adaptive Capacity or recovery (no perturbation in these data), insurance or employment decisions, or generalization to Peruvian or other non-US populations without validation. Risks: the methylation subsample is not representative of the US population without weights; mortality prediction in people aged 50 or older only; follow-up times in the public mortality file are partly synthetic.""",
 "6. Distribution": """The source files are public-use files released by the CDC/NCHS, downloaded directly from CDC servers (wwwn.cdc.gov and ftp.cdc.gov); this repository redistributes no data, only code and results. NCHS terms for the linked mortality files: statistical reporting and analysis only, no re-identification, and publications must cite NCHS as the source with the NCHS disclaimer; the CpG-level methylation matrix is restricted and not used. License of the derived code and cards: UNKNOWN — to investigate. DOI for this repository: UNKNOWN — to investigate (Zenodo planned). Export controls or third-party restrictions: none known for these public-use files.""",
 "7. Maintenance": """Maintainer: Aly Gallo Lopez (contact through the repository). Updates: none scheduled; the notebook is re-run if the CDC changes file locations (the download cell stops with a message listing failed files). Erratum policy: corrections are made as new commits, with earlier versions kept in the repository history. Mortality follow-up is fixed to 2019; extensions of follow-up: UNKNOWN — to investigate."""}

def render_sheet(sheet):
    out = ["# Datasheet — NHANES 1999-2002 analysis file (age >= 50, DNA methylation, linked mortality)\n"]
    for k, v in sheet.items():
        out.append(f"## {k}\n\n{v if v and str(v).strip() else 'UNKNOWN — to investigate'}\n")
    return "\n".join(out)
DATASHEET_MD = render_sheet(SHEET)
open("DATASHEET.md", "w").write(DATASHEET_MD)
print(DATASHEET_MD[:2500], "\n...\n[written to DATASHEET.md]")


# Datasheet — NHANES 1999-2002 analysis file (age >= 50, DNA methylation, linked mortality)

## 1. Motivation

Purpose: to test whether a composite of routine biomarkers (reduced BSI) adds mortality prognosis over age, sex, an epigenetic clock and allostatic load, as an exploratory step of a doctoral project (ChroniCare-AI, UNMSM). We did not create NHANES; we assembled an analysis file from three public CDC releases. Original creators: CDC National Center for Health Statistics (NHANES, DNA-methylation epigenetic biomarkers release, Linked Mortality Files). Funding of the original data: US federal. Funding of this analysis: UNKNOWN — to investigate (self-funded doctoral work).

## 2. Composition

Instances: NHANES participants. Cycles 1999-2000 and 2001-2002 (examination files DEMO/DEMO_B and laboratory files), adults aged 50 or older in the CDC DNA-methylation release with eligible linked mortality: 4,446 rows in the merged files (2,493 deaths); the CDC documentation states that parti

## 8. Exercise 6 — Documentation scorecard (8 checks)
The eight checks come from the class pair-feedback checklist and the grading rubric (completeness, honesty, disaggregation, reproducibility). They are our own implementation, not the official notebook function. Each is Y/N with the evidence found; "UNKNOWN" gaps are counted, not hidden.

In [14]:
import re
def score(card_md, sheet_md):
    c, s = card_md.lower(), sheet_md.lower()
    n9 = len(re.findall(r"^## \d\. ", card_md, flags=re.M)); n7 = len(re.findall(r"^## \d\. ", sheet_md, flags=re.M))
    checks = [
     ("1. Model Card has all 9 sections", n9 == 9, f"{n9} sections"),
     ("2. Datasheet has all 7 sections", n7 == 7, f"{n7} sections"),
     ("3. Intended use AND misuse/out-of-scope stated", "intended use" in c and "out-of-scope" in c and "misuse" in c, "section 2"),
     ("4. Metrics broken down by at least one subgroup", "subgroups (out-of-fold" in c and ("sex: " in c) and ("age 65+" in c), "section 7"),
     ("5. Datasheet says how data were collected and by whom", "collected by the cdc" in s, "section 3"),
     ("6. Honest 'should NOT be used for' line", "should not be used for" in s and "must not be used" in c, "datasheet section 5, card section 8"),
     ("7. Gaps flagged as UNKNOWN (not left blank)", (c + s).count("unknown — to investigate") >= 3, f"{(c + s).count('unknown — to investigate')} flagged gaps"),
     ("8. A stranger could reproduce it (data URLs, seeds, versions, one-click run)", all(k in c + s for k in ["wwwn.cdc.gov", "seeds", "lifelines"]), "download cell, 10 seeds, versions printed"),
    ]
    tab = pd.DataFrame([{"check": a, "pass": "Y" if ok else "N", "evidence": ev} for a, ok, ev in checks]); display(tab)
    print(f"SCORE: {sum(ok for _, ok, _ in checks)}/8")
    return tab
SCORECARD = score(MODEL_CARD_MD, DATASHEET_MD)
print("Note: 8/8 means the cards pass these formal checks. It does not prove the content is right or complete; the gaps are listed as UNKNOWN in the cards and are part of the honest record.")


,check,pass,evidence
0,1. Model Card has all 9 sections,Y,9 sections
1,2. Datasheet has all 7 sections,Y,7 sections
2,3. Intended use AND misuse/out-of-scope stated,Y,section 2
3,4. Metrics broken down by at least one subgroup,Y,section 7
4,5. Datasheet says how data were collected and ...,Y,section 3
5,6. Honest 'should NOT be used for' line,Y,"datasheet section 5, card section 8"
6,7. Gaps flagged as UNKNOWN (not left blank),Y,7 flagged gaps
7,"8. A stranger could reproduce it (data URLs, s...",Y,"download cell, 10 seeds, versions printed"


SCORE: 8/8
Note: 8/8 means the cards pass these formal checks. It does not prove the content is right or complete; the gaps are listed as UNKNOWN in the cards and are part of the honest record.


In [15]:
# Download the two Markdown files (they also stay in the Colab file panel)
try:
    from google.colab import files
    files.download("MODEL_CARD.md"); files.download("DATASHEET.md")
except Exception as ex:
    print("Not in Colab; files are in the working directory:", ex)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>